# Bài 2 — Quan sát và kiểm soát chi phí ⭐⭐

**Tuần 22** · Bài tập kèm theo: `ex03_van_hanh.py`

Đưa được ứng dụng lên internet là phần dễ. Phần khó là trả lời được ba câu:

1. **Nó đang chậm hay nhanh?** — và chậm với *bao nhiêu phần trăm* người dùng?
2. **Tháng này hết bao nhiêu tiền?** Còn bao nhiêu?
3. **Khi Anthropic gián đoạn, ứng dụng của bạn làm gì?**

Không cần API key. Và **không dùng `sleep()`** — mọi thứ ở đây nhận đồng hồ làm
tham số.

In [ ]:
import random
import sys
from pathlib import Path

GOC = Path.cwd()
while not (GOC / "pyproject.toml").exists() and GOC != GOC.parent:
    GOC = GOC.parent
sys.path.insert(0, str(GOC / "curriculum" / "10-deploy-ops" / "solutions"))

for _l in (sys.stdout, sys.stderr):
    if (getattr(_l, "encoding", "") or "").lower().replace("-", "") != "utf8":
        try:
            _l.reconfigure(encoding="utf-8", errors="replace")
        except Exception:
            pass

from ex03_van_hanh import (BoDem, GioiHanTanSuat, NgatMach, SoChiPhi,
                           phan_vi, tao_ban_ghi_trace)


class DongHo:
    """Đồng hồ giả lập - test và demo không phải chờ đợi thật."""

    def __init__(self, bat_dau=0.0):
        self.luc = float(bat_dau)

    def __call__(self):
        return self.luc

    def tien(self, giay):
        self.luc += giay


print("Sẵn sàng.")

## 1. Vì sao trung bình nói dối

Đây là số liệu của một dịch vụ hoàn toàn bình thường: hầu hết request nhanh,
vài request rất chậm.

In [ ]:
random.seed(42)

# 950 request nhanh, 50 request chậm (retry, cold start, prompt dài)
do_tre = [random.gauss(200, 40) for _ in range(950)] + \
         [random.gauss(8000, 2000) for _ in range(50)]
do_tre = [max(10.0, d) for d in do_tre]

trung_binh = sum(do_tre) / len(do_tre)

print(f"Số request      : {len(do_tre)}")
print(f"TRUNG BÌNH      : {trung_binh:>8.0f} ms   <- nghe ổn")
print(f"p50 (trung vị)  : {phan_vi(do_tre, 50):>8.0f} ms")
print(f"p95             : {phan_vi(do_tre, 95):>8.0f} ms   <- 5% người dùng chịu cái này")
print(f"p99             : {phan_vi(do_tre, 99):>8.0f} ms")
print(f"Chậm nhất       : {max(do_tre):>8.0f} ms")

Nhìn kỹ: **trung bình còn cao hơn cả p95.** Nó không mô tả một người dùng nào
cả — 95% nhanh hơn thế nhiều, và 5% còn lại chậm hơn thế rất nhiều.

> 📌 **Độ trễ là một phân bố lệch phải.** Trung bình chỉ mô tả được phân bố đối
> xứng — mà độ trễ thì không bao giờ đối xứng.
>
> Và 5% người dùng chậm đó không phải 5% ngẫu nhiên mỗi lần. Họ thường là những
> người có prompt dài nhất, tức là **người dùng nặng nhất của bạn**.

> ⚠️ **Nhưng bảng trên còn dạy một điều nữa — và người viết notebook này cũng
> suýt bỏ sót:** p95 ở đây chỉ ~350 ms, trông rất đẹp. Cái đuôi 10 giây chỉ lộ
> ra ở **p99**.
>
> Lý do đơn giản: đúng 5% request chậm, nên p95 rơi ngay vào ranh giới và vẫn
> nằm bên phần nhanh. **Chọn nhầm phân vị thì bạn không thấy gì cả.**
>
> Nguyên tắc thực dụng: báo cáo **p50, p95 và p99 cùng nhau**. Một con số duy
> nhất — dù là trung bình hay p95 — đều giấu được một sự thật khó chịu.

In [ ]:
# Thấy rõ hơn bằng biểu đồ cột
buc = [0, 250, 500, 1000, 2000, 5000, 10000, 999999]
ten = ["<250ms", "250-500", "500ms-1s", "1-2s", "2-5s", "5-10s", ">10s"]

print("Phân bố độ trễ:")
for i, nhan in enumerate(ten):
    n = sum(1 for d in do_tre if buc[i] <= d < buc[i + 1])
    print(f'  {nhan:<10} {"█" * max(0, round(n / 20)):<50} {n:>4}')

## 2. Bộ đếm

In [ ]:
bd = BoDem()
random.seed(7)
for d in do_tre[:200]:
    thanh_cong = random.random() > 0.03          # ~3% lỗi
    bd.ghi(d, thanh_cong, chi_phi_usd=0.0012 if thanh_cong else 0.0)

t = bd.tom_tat()
print(f'{"số request":<16}{t["so_request"]:>10}')
print(f'{"số lỗi":<16}{t["so_loi"]:>10}')
print(f'{"tỷ lệ lỗi":<16}{t["ty_le_loi"]:>10.1%}')
print(f'{"p50":<16}{t["p50_ms"]:>10.0f} ms')
print(f'{"p95":<16}{t["p95_ms"]:>10.0f} ms')
print(f'{"tổng chi phí":<16}{t["tong_chi_phi"]:>10.4f} USD')
print(f'{"chi phí / req":<16}{t["chi_phi_tb"]:>10.5f} USD')

> ⚠️ **`ty_le_loi` phải chia cho TỔNG số request, không phải số lỗi.** Chia
> nhầm cho số lỗi thì dashboard của bạn luôn hiển thị 100% — và điều buồn cười là
> lỗi này tồn tại rất lâu, vì "100% lỗi" trông giống một sự cố thật đến mức không
> ai nghi ngờ chính cái công thức.

Bộ đếm này giữ **mọi giá trị trong bộ nhớ**. Với dịch vụ chạy liên tục thì đó là
rò rỉ bộ nhớ — sau một tuần bạn có vài triệu phần tử. Cách làm thật: histogram có
sẵn khoảng (Prometheus), hoặc cửa sổ trượt chỉ giữ N giá trị gần nhất.

Giữ bản đơn giản ở đây là **cố ý**, nhưng phải biết giới hạn của nó.

## 3. Ngân sách: chặn trước khi hết tiền

In [ ]:
so = SoChiPhi(ngan_sach_usd=1.0, nguong_canh_bao=0.8)

random.seed(1)
for i in range(1, 900):
    chi_phi_du_kien = 0.0012
    if not so.cho_phep(chi_phi_du_kien):
        print(f"⛔ Request thứ {i}: TỪ CHỐI - hết ngân sách")
        break
    # Chi phí THẬT chỉ biết SAU khi gọi xong (dựa trên usage trả về)
    so.ghi_nhan(chi_phi_du_kien * random.uniform(0.7, 1.4))

    tt = so.trang_thai()
    if tt["can_canh_bao"] and i % 50 == 0:
        print(f'⚠️  Request {i}: đã dùng {tt["ty_le"]:.0%} ngân sách, '
              f'còn ${tt["con_lai"]:.3f}')

> 📌 **Vì sao `cho_phep` và `ghi_nhan` là hai hàm riêng?**
>
> Trước khi gọi API, bạn chỉ **ước lượng** được chi phí. Con số thật chỉ có sau
> khi gọi xong, dựa trên `usage` trả về. Gộp làm một hàm nghĩa là hoặc bạn chặn
> dựa trên số liệu sai, hoặc bạn ghi nhận một chi phí chưa xảy ra.
>
> Vòng lặp trên mô phỏng đúng điều đó: kiểm bằng ước lượng, ghi nhận bằng con số
> thật (dao động 0.7–1.4 lần ước lượng).

## 4. Giới hạn tần suất: token bucket

In [ ]:
dh = DongHo()
gh = GioiHanTanSuat(suc_chua=5, toc_do_nap=1.0, ham_gio=dh)

print("10 request liên tiếp (xô đầy 5 token):")
print("  ", "".join("✅" if gh.cho_phep() else "❌" for _ in range(10)))

print()
print("Sau 3 giây (nạp thêm 3 token):")
dh.tien(3.0)
print("  ", "".join("✅" if gh.cho_phep() else "❌" for _ in range(5)))

print()
print("Sau 1 giờ - xô KHÔNG tràn quá sức chứa:")
dh.tien(3600)
print("  ", "".join("✅" if gh.cho_phep() else "❌" for _ in range(8)))

Dòng cuối là điểm mấu chốt: nghỉ một tiếng **không** cho phép bạn gửi 3.600
request cùng lúc. Xô có trần.

> 📌 **Vì sao token bucket mà không phải "tối đa N request mỗi phút"?**
>
> Đếm theo cửa sổ cố định cho phép gấp đôi tải ở ranh giới: 100 request lúc
> 10:00:59 và 100 request nữa lúc 10:01:00 đều "hợp lệ" — nhưng máy chủ của bạn
> nhận 200 request trong một giây.
>
> Token bucket không có ranh giới để mà lách.

## 5. Ngắt mạch: khi dịch vụ phụ thuộc gián đoạn

Đây là phần mà người mới hay bỏ qua, và nó là khác biệt giữa *"dịch vụ của tôi
chậm khi Anthropic gián đoạn"* và *"dịch vụ của tôi chết theo"*.

In [ ]:
dh = DongHo()
nm = NgatMach(nguong_loi=3, thoi_gian_cho=30.0, ham_gio=dh)

def thu_goi(mo_ta, se_that_bai):
    if not nm.cho_goi():
        print(f'{mo_ta:<34} ⛔ TỪ CHỐI NGAY (mạch {nm.trang_thai})')
        return
    if se_that_bai:
        nm.ghi_that_bai()
        print(f'{mo_ta:<34} ❌ thất bại  -> mạch {nm.trang_thai}')
    else:
        nm.ghi_thanh_cong()
        print(f'{mo_ta:<34} ✅ thành công -> mạch {nm.trang_thai}')

thu_goi("Request 1 (Anthropic hỏng)", True)
thu_goi("Request 2 (Anthropic hỏng)", True)
thu_goi("Request 3 (Anthropic hỏng)", True)
thu_goi("Request 4", True)
thu_goi("Request 5", True)

**Ba request đầu mỗi cái chờ 30 giây rồi mới thất bại. Request thứ tư trở đi
thất bại ngay lập tức.**

Đó là toàn bộ giá trị của ngắt mạch: nó biến 30 giây chờ đợi vô ích thành một lỗi
trả về tức thì, và **giữ lại tài nguyên** cho những việc còn chạy được.

In [ ]:
print("Sau 30 giây, mạch tự chuyển sang 'thử lại':")
dh.tien(31.0)
print(f"   trạng thái: {nm.trang_thai}")
print()

thu_goi("Request thử lại (vẫn hỏng)", True)
print("   -> mở lại NGAY, không cần đếm đủ 3 lần nữa")
print()

dh.tien(31.0)
thu_goi("Request thử lại (đã hồi phục)", False)

> 📌 **Trạng thái được TÍNH, không được LƯU.**
>
> Nếu bạn lưu `"thu_lai"` vào một biến, phải có ai đó gọi để cập nhật nó đúng lúc
> — một bộ đếm nền, một vòng lặp kiểm tra. Tính lại mỗi lần đọc thì không bao giờ
> lệch, và không cần thành phần nào khác.
>
> Đây là một nguyên tắc rộng hơn ngắt mạch: **trạng thái suy ra được từ dữ liệu
> có sẵn thì đừng lưu nó.** Mỗi bản sao là một cơ hội để hai bên lệch nhau.

## 6. Trace: nối các dòng log của cùng một request

In [ ]:
import json
import uuid

trace_id = uuid.uuid4().hex[:12]

cac_su_kien = [
    tao_ban_ghi_trace(trace_id, "nhan_request", duong_dan="/hoi", do_dai_cau_hoi=42),
    tao_ban_ghi_trace(trace_id, "goi_model", model="claude-haiku-4-5", max_token=1024),
    tao_ban_ghi_trace(trace_id, "model_tra_ve", token_vao=180, token_ra=95,
                      do_tre_ms=1240),
    tao_ban_ghi_trace(trace_id, "tra_ve", ma_http=200, tong_do_tre_ms=1310,
                      chi_phi_usd=0.00066),
]

for e in cac_su_kien:
    print(json.dumps(e, ensure_ascii=False))

Bốn dòng này ghi ra ở bốn thời điểm khác nhau, xen kẽ với log của hàng trăm
request khác. `trace_id` là thứ duy nhất nối chúng lại.

> 📌 **Log có cấu trúc, không phải chuỗi.** Với chuỗi, câu hỏi *"request nào chậm
> hơn 2 giây?"* trở thành một bài toán regex. Với JSON, nó là một phép lọc.

Và chú ý dấu `/` trong chữ ký `tao_ban_ghi_trace(trace_id, su_kien, /, **truong)`:
nó làm hai tham số đầu **positional-only**, nhờ đó lời gọi `trace_id="giả"` rơi
vào `**truong` và bị bắt. Không có `/`, Python chặn trước bằng `TypeError` và
nhánh kiểm tra của bạn **không bao giờ chạy** — một quy tắc chết mà không ai nhận ra.

## 7. Ghép tất cả lại

In [ ]:
class DichVu:
    """Gói toàn bộ lớp vận hành quanh một lời gọi model."""

    def __init__(self, goi_model, ngan_sach_usd=1.0, ham_gio=None):
        self.goi_model = goi_model
        self.bo_dem = BoDem()
        self.so = SoChiPhi(ngan_sach_usd)
        self.gioi_han = GioiHanTanSuat(10, 5.0, ham_gio)
        self.ngat_mach = NgatMach(3, 30.0, ham_gio)
        self.ham_gio = ham_gio or (lambda: 0.0)

    def hoi(self, cau_hoi, chi_phi_du_kien=0.0012):
        trace_id = uuid.uuid4().hex[:8]

        # Thứ tự bốn cửa: rẻ nhất và cục bộ nhất trước
        if not self.gioi_han.cho_phep():
            return {"ma_loi": "qua_nhieu_yeu_cau", "trace_id": trace_id}
        if not self.so.cho_phep(chi_phi_du_kien):
            return {"ma_loi": "het_ngan_sach", "trace_id": trace_id}
        if not self.ngat_mach.cho_goi():
            return {"ma_loi": "dich_vu_gian_doan", "trace_id": trace_id}

        bat_dau = self.ham_gio()
        try:
            tra_loi = self.goi_model(cau_hoi)
        except Exception:
            self.ngat_mach.ghi_that_bai()
            self.bo_dem.ghi((self.ham_gio() - bat_dau) * 1000, False)
            return {"ma_loi": "loi_he_thong", "trace_id": trace_id}

        self.ngat_mach.ghi_thanh_cong()
        self.so.ghi_nhan(chi_phi_du_kien)
        self.bo_dem.ghi((self.ham_gio() - bat_dau) * 1000, True, chi_phi_du_kien)
        return {"tra_loi": tra_loi, "trace_id": trace_id}


dh = DongHo()
# Ngân sách rộng rãi để thấy RÕ cửa giới hạn tần suất (xô 10 token)
dich_vu = DichVu(lambda q: f"Trả lời: {q}", ngan_sach_usd=10.0, ham_gio=dh)

print("15 request liên tiếp - xô chỉ có 10 token:")
for i in range(15):
    kq = dich_vu.hoi(f"câu {i}")
    dau = "✅" if "tra_loi" in kq else f'⛔ {kq["ma_loi"]}'
    print(f"  {i + 1:>2}. {dau}")

Mười request đầu đi qua, request thứ 11 trở đi bị chặn — xô đã cạn và
chưa kịp nạp lại vì đồng hồ không nhúc nhích.

**Thứ tự bốn cửa không tuỳ tiện:** giới hạn tần suất → ngân sách → ngắt mạch
→ gọi thật. Mỗi cửa đắt hơn cửa trước, nên cửa rẻ nhất phải đứng đầu.

Kiểm tra tần suất là một phép trừ; gọi API là một vòng mạng mất một giây. Đặt sai
thứ tự nghĩa là bạn trả tiền cho những request lẽ ra đã bị từ chối miễn phí.

In [ ]:
# Dịch vụ MỚI, ngân sách hẹp - lần này để thấy cửa ngân sách
dh2 = DongHo()
dv2 = DichVu(lambda q: f"Trả lời: {q}", ngan_sach_usd=0.012, ham_gio=dh2)

so_thanh_cong = 0
for i in range(50):
    dh2.tien(1.0)                     # nghỉ 1 giây giữa mỗi request -> không đụng tần suất
    kq = dv2.hoi(f"câu {i}")
    if "tra_loi" in kq:
        so_thanh_cong += 1
    elif kq["ma_loi"] == "het_ngan_sach":
        print(f"⛔ Dừng ở request {i + 1}: hết ngân sách sau {so_thanh_cong} lần gọi")
        break

t = dv2.bo_dem.tom_tat()
tt = dv2.so.trang_thai()
print()
print(f'Đã phục vụ : {t["so_request"]} request, tỷ lệ lỗi {t["ty_le_loi"]:.0%}')
print(f'Đã tiêu    : ${tt["da_dung"]:.4f} / ${dv2.so.ngan_sach:.4f}  ({tt["ty_le"]:.0%})')
print(f'Còn lại    : ${tt["con_lai"]:.4f}')

## Tóm tắt

| Điều | Nhớ gì |
|---|---|
| **p95, không phải trung bình** | Độ trễ lệch phải; trung bình không mô tả ai cả |
| **`ty_le_loi`** | Chia cho TỔNG request. Chia nhầm thì dashboard luôn 100% |
| **`cho_phep` ≠ `ghi_nhan`** | Trước khi gọi chỉ ước lượng được; sau mới biết thật |
| **Token bucket** | Không có ranh giới cửa sổ để mà lách |
| **Ngắt mạch** | Biến 30 giây chờ vô ích thành lỗi tức thì |
| **Trạng thái tính, đừng lưu** | Mỗi bản sao là một cơ hội để lệch |
| **Trace ID** | Thứ duy nhất nối các dòng log của cùng một request |
| **Thứ tự các cửa** | Rẻ nhất trước — đừng trả tiền cho request đã bị từ chối |

**Bài tập:** `curriculum/10-deploy-ops/exercises/ex03_van_hanh.py` ⭐⭐

```powershell
pytest tests/phase10/test_ex03.py -v
```

➡️ Bài tiếp: **[03 — Đóng gói và CI](03_dong_goi_va_ci.ipynb)**